In [1]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

from pyspark.sql import Window

from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, MinHashLSH

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

print("Spark version:", spark.version)

Spark version: 3.5.1


In [2]:
reviews_path = "gs://msca-bdp-students-bucket/kireetij_final/reviews_clean"
meta_path    = "gs://msca-bdp-students-bucket/kireetij_final/meta_clean"

df_reviews = spark.read.parquet(reviews_path)
df_meta    = spark.read.parquet(meta_path)

print("Reviews:", df_reviews.count())
print("Meta:", df_meta.count())

df_reviews.printSchema()
df_meta.printSchema()

Reviews: 59966506
Meta: 1751297
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- review_ts: timestamp (nullable = true)
 |-- review_date: date (nullable = true)

root
 |-- average_rating: double (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- title: string (nullable = true)
 |-- price_num: double (nullable = true)



In [3]:
CHOSEN_CATEGORY = "Books"

df_meta_named = df_meta.select(
    "parent_asin",
    "main_category",
    F.col("title").alias("product_title"),
    "price_num"
)

df_joined = df_reviews.join(
    df_meta_named,
    on="parent_asin",
    how="left"
)

df_cat = df_joined.filter(F.col("main_category") == CHOSEN_CATEGORY)

print("Total reviews in chosen category:", df_cat.count())

df_cat_sample = (
    df_cat
    .filter(F.col("text").isNotNull())
    .filter(F.length(F.col("text")) > 20)   # drop ultra-short texts
    .withColumn("review_id", F.monotonically_increasing_id())  # unique ID for similarity pairs
)

print("Sample size in chosen category (before any further sampling):", df_cat_sample.count())
df_cat_sample.select("review_id", "parent_asin", "title", "text").show(5, truncate=80)

Total reviews in chosen category: 674


Sample size in chosen category (before any further sampling): 564


+----------+-----------+-------------------------------------------------------+--------------------------------------------------------------------------------+
| review_id|parent_asin|                                                  title|                                                                            text|
+----------+-----------+-------------------------------------------------------+--------------------------------------------------------------------------------+
|         0| 0446581348|                                             Must Have.|This is the real makeup bible. Everything you want to know you'll found on it...|
|         1| 0446581348|                                             Five Stars|                            My friend loves this book. I need to buy one for me!|
|         2| 1563923572|                                             Five Stars|                                                     Just as described.  Thanks.|
|         3| 0446581348|    

In [4]:
cat_sample_path = "gs://msca-bdp-students-bucket/kireetij_final/df_cat_sample"

(
    df_cat_sample
    .write
    .mode("overwrite")
    .parquet(cat_sample_path)
)

print("Saved df_cat_sample to:", cat_sample_path)

Saved df_cat_sample to: gs://msca-bdp-students-bucket/kireetij_final/df_cat_sample
